# EvoSim Analysis (Colab Compatible)

Run this notebook to reproduce the findings in `docs/EVOSIM_PAPER.md`.

## 0. Colab Setup
If you are running this in Google Colab, execute this cell to clone the repository and get the data.

In [ ]:
import os
try:
    import google.colab
    if not os.path.exists('evosim-agentic-sociology'):
        print("Detected Google Colab. Cloning repository...")
        !git clone https://github.com/HarperKollins/evosim-agentic-sociology.git
    
    os.chdir('evosim-agentic-sociology')
    if os.path.exists('datasets.zip'):
        print("Unzipping datasets.zip...")
        !unzip -o datasets.zip
    else:
        print("Warning: datasets.zip not found!")
        
    os.chdir('analysis')
    print("Setup complete. Working directory: analysis/")
except ImportError:
    print("Not running in Colab. Assuming local environment.")

## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import glob

# Load Simulation Data
print("Looking for CSVs in: ../experiments/*.csv and ../*.csv")
sim_files = glob.glob('../experiments/*.csv') + glob.glob('../*.csv')
sim_dfs = []
for f in sim_files:
    if 'sim' in f or 'simulation' in f: 
        try: 
             df = pd.read_csv(f)
             df['Source_File'] = os.path.basename(f)
             sim_dfs.append(df)
        except: pass

if not sim_dfs:
    print("Error: No simulation CSV files found. Please check your file paths and names.")
else:
    full_sim = pd.concat(sim_dfs, ignore_index=True)
    print(f"Loaded {len(full_sim)} simulation rows from {len(sim_dfs)} files.")

## 2. The Tribal Imperative
Calculates survival diff between Tribal vs Lone agents.

In [ ]:
if sim_dfs:
    id_col = 'ID' if 'ID' in full_sim.columns else 'AgentId'
    agents = full_sim.sort_values('Tick').groupby(['Source_File', id_col]).last().reset_index()

    if 'TribeId' in agents.columns:
        tribal = agents[agents['TribeId'] != -1]['Age']
        lone = agents[agents['TribeId'] == -1]['Age']
        
        t_stat, p_val = stats.ttest_ind(tribal, lone, equal_var=False)
        print(f"Tribal Mean Age: {tribal.mean():.2f} (N={len(tribal)})")
        print(f"Lone Mean Age: {lone.mean():.2f} (N={len(lone)})")
        print(f"P-Value: {p_val:.5e}")
        
        plt.figure(figsize=(8, 6))
        sns.boxplot(x=agents['TribeId'] != -1, y=agents['Age'])
        plt.xticks([0, 1], ['Lone Wolf', 'Tribal Member'])
        plt.title('Survival Advantage of Tribalism')
        plt.show()

## 3. The Breaking Bad Hypothesis
Correlates Age with Karma.

In [ ]:
if sim_dfs and 'Karma' in agents.columns:
    valid = agents[agents['Age'] > 5]
    corr, p_val = stats.pearsonr(valid['Age'], valid['Karma'])
    print(f"Correlation: {corr:.3f}")
    print(f"P-Value: {p_val:.5e}")
    
    plt.figure(figsize=(10, 6))
    sns.regplot(x='Age', y='Karma', data=valid, 
                scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
    plt.title('Age vs Karma')
    plt.show()